# Comparison with existing identifier mappers (Figure 4d)

This notebook builds a **capability-first** (not accuracy-first) comparison between IDTrack and common point-in-time identifier mappers.

## The claim this supports (marketing without overclaim)

Most identifier mappers behave like: “give me the best *current* answer”. For convenience lookups this is excellent.
For **reproducible atlas-scale integration** and long-lived pipelines, it is incomplete because identifier identity is an implicit coordinate in:

`namespace × Ensembl time (release) × genome assembly`

IDTrack’s marketing edge is not “always a better answer” but **a better contract**:

- You can **bound evidence in time** (snapshot boundary) so “which Ensembl history window?” becomes a reportable parameter
- You can treat releases as a **time axis** (time travel), not a hidden dependency on “latest”
- You get explicit **1→0 / 1→1 / 1→n** outcomes so ambiguity is reportable rather than silently coerced
- You can request **audit payloads** (`explain=True`) so mappings become inspectable objects, not black-box strings

This notebook contrasts those *reproducibility controls* against widely used point-in-time tools.

## What this notebook produces

- Figure candidate: `_outputs/_publication/figures/fig_tool_comparison_matrix.pdf`
- Table export: `_outputs/_publication/tables/tool_comparison_capability_matrix.csv`
- Table export: `_outputs/_publication/tables/tool_comparison_capability_matrix.tex`
- Cached external-mapper demo outputs (pickles + CSV summaries) under `idtrack/docs/_notebooks/idtrack_cache/experiments/comparison/`

## Scope and guardrails (important)

- This is **not** an accuracy benchmark (no precision/recall claims).
- This notebook avoids speed claims; external APIs vary and rate-limit.
- Demo queries are intentionally small and cached to avoid repeatedly hitting public services.
- IDTrack conversions are not executed here (graph loading is heavy). For IDTrack outcome panels see:
  - `experiment_random_stress_tests/00_random_stress_tests_fig4ab.ipynb`
  - `experiment_hlca/00_hlca_manuscript_table1_and_figures.ipynb`

## Expected results (what “good” looks like)

- The capability matrix should cleanly separate “point-in-time mapping” from “snapshot-bounded time-travel mapping”.
- Even on a tiny demo set, external mappers should show non-trivial 1→0 and occasional 1→n, and backends may disagree.

## Reproducibility knobs worth reporting (Methods-facing)

- IDTrack: `snapshot_release`, `to_release`, `final_database`, `strategy`, and the external YAML allowlist
- External mappers: whether you can pin a historical release (pybiomart can; most services cannot)


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:  # noqa: S110
    sns = None

import os
import sys

# Add experiments/src to sys.path (layout-aware; works on Slurm and locally)
REPO_ROOT = Path(os.environ.get('REPO_ROOT', Path.cwd())).expanduser().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (
    (REPO_ROOT / 'idtrack').is_dir()
    and ((REPO_ROOT / 'reproducibility').is_dir() or (REPO_ROOT / 'idtrack' / 'reproducibility').is_dir())
):
    REPO_ROOT = REPO_ROOT.parent

REPRO_ROOT = REPO_ROOT / 'reproducibility' if (REPO_ROOT / 'reproducibility').is_dir() else REPO_ROOT / 'idtrack' / 'reproducibility'
EXPERIMENTS_SRC = REPRO_ROOT / 'experiments' / 'src'
if str(EXPERIMENTS_SRC) not in sys.path:
    sys.path.insert(0, str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    MANUSCRIPT_COLORS,
    notebook_context,
    read_pickle,
    save_figure,
    safe_tag,
    write_pickle,
)

ctx = notebook_context('comparison', start=REPO_ROOT)
plt.rcParams.update({'savefig.dpi': 300})

IDTRACK_LOCAL_REPO = ctx.idtrack_local_repo
CACHE_DIR = ctx.experiment_cache
MANUSCRIPT_FIGURES = ctx.manuscript_figures

print('Repo root:', REPO_ROOT)
print('IDTRACK_LOCAL_REPO:', IDTRACK_LOCAL_REPO)
print('CACHE_DIR:', CACHE_DIR)


In [ ]:
# -------------------- Configure comparison --------------------

# External mapper methods (optional deps)
METHODS = ['pybiomart', 'mygene', 'gprofiler', 'gget']

# Demo query sets (small on purpose; safe for public APIs)
# - includes a version-suffixed Ensembl ID (to exercise version stripping)
# - includes a negative control (to guarantee at least one 1→0 outcome)
QUERY_ENSG = [
    'ENSG00000139618',  # BRCA2
    'ENSG00000141510',  # TP53
    'ENSG00000157764',  # BRAF
    'ENSG00000121879',  # KRAS
    'ENSG00000171862',  # PTEN
    'ENSG00000136997',  # MYC
    'ENSG00000146648',  # EGFR
    'ENSG00000141510.18',  # TP53 (versioned)
    'ENSG_DOES_NOT_EXIST',  # negative control
]

SPECIES = 'human'

QUERY_SYMBOLS = [
    'BRCA2',
    'TP53',
    'BRAF',
    'KRAS',
    'PTEN',
    'MYC',
    'EGFR',
    'SYMBOL_DOES_NOT_EXIST',
]

QUERY_UNIPROT = [
    'P51587',  # BRCA2
    'P04637',  # TP53
    'P15056',  # BRAF
    'P01116',  # KRAS
    'P60484',  # PTEN
    'P01106',  # MYC
    'P00533',  # EGFR
    'UNIPROT_DOES_NOT_EXIST',
]

# pybiomart only: pin an explicit historical release.
# (Other services usually do not expose this as a stable control.)
PYBIOMART_RELEASE = 107

# Scenarios keep the focus on manuscript-relevant targets: HGNC, Ensembl gene IDs, UniProt.
SCENARIOS = [
    {
        'label': 'Ensembl→HGNC',
        'ids': QUERY_ENSG,
        'input_db': 'ensembl_gene',
        'output_db': 'HGNC Symbol',
    },
    {
        'label': 'Ensembl→UniProt',
        'ids': QUERY_ENSG,
        'input_db': 'ensembl_gene',
        'output_db': 'UniProtKB/Swiss-Prot',
    },
    {
        'label': 'HGNC→Ensembl',
        'ids': QUERY_SYMBOLS,
        'input_db': 'HGNC Symbol',
        'output_db': 'ensembl_gene',
    },
    {
        'label': 'UniProt→Ensembl',
        'ids': QUERY_UNIPROT,
        'input_db': 'UniProtKB/Swiss-Prot',
        'output_db': 'ensembl_gene',
    },
]

print('Methods:', METHODS)
print('Scenarios:', [s['label'] for s in SCENARIOS])
print('pybiomart release pin:', PYBIOMART_RELEASE)


In [ ]:
# -------------------- External mapper availability --------------------

import idtrack._external_mappers as ext

status = ext.check_optional_dependencies(warn=True)

pd.DataFrame([{'dependency': k, 'installed': v} for k, v in status.items()]).sort_values('dependency').reset_index(drop=True)


In [ ]:
# -------------------- Run external mappers (cached) --------------------

import hashlib

import idtrack._external_mappers as ext


def _ids_fingerprint(ids: list[str]) -> str:
    # Used only for cache file naming.
    h = hashlib.md5('|'.join(map(str, ids)).encode('utf-8')).hexdigest()  # noqa: S324
    return h[:12]


def _cache_path(label: str, method: str, input_db: str, output_db: str, ids: list[str]) -> Path:
    fp = _ids_fingerprint(ids)
    tag = (
        f"extmap_{safe_tag(label)}_{method}_in{safe_tag(input_db)}_out{safe_tag(output_db)}_"
        f"n{len(ids)}_{fp}_species{safe_tag(SPECIES)}_pybiomart{PYBIOMART_RELEASE}.pickle"
    )
    return CACHE_DIR / tag


def _normalize(df: pd.DataFrame | None) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame(columns=['input_id', 'output_id', 'mapping'])
    x = df.copy()
    # Normalize to a stable schema for downstream counting + exports
    cols = [
        c
        for c in ['input_id', 'output_id', 'mapping', 'method', 'input_db', 'output_db', 'release_used']
        if c in x.columns
    ]
    return x[cols].copy()


def _counts_1_to_0_1_to_1_1_to_n(df: pd.DataFrame, inputs: list[str]) -> dict[str, int]:
    if df is None or df.empty:
        return {'1→0': len(set(inputs)), '1→1': 0, '1→n': 0}
    per = df.drop_duplicates('input_id')
    counts = per['mapping'].value_counts().to_dict()
    return {
        '1→0': int(counts.get('1:0', 0)),
        '1→1': int(counts.get('1:1', 0)),
        '1→n': int(counts.get('1:n', 0)),
    }


results: dict[tuple[str, str], pd.DataFrame] = {}
errors: list[dict] = []

for scenario in SCENARIOS:
    label = str(scenario['label'])
    input_db = str(scenario['input_db'])
    output_db = str(scenario['output_db'])
    ids = [str(x) for x in (scenario['ids'] or [])]

    for method in METHODS:
        p = _cache_path(label, method, input_db, output_db, ids)
        if p.exists():
            df = read_pickle(p)
            results[(label, method)] = _normalize(df)
            print('Loaded:', p.name)
            continue

        try:
            kwargs = {
                'ids': ids,
                'input_db': input_db,
                'output_db': output_db,
                'method': method,
                'species': SPECIES,
                'chunk_size': 200,
                'pause': 0.1,
                'verbose': 2,
            }
            if method == 'pybiomart':
                kwargs['release_for_pybiomart'] = PYBIOMART_RELEASE

            df = ext.convert_ids(**kwargs)
            df = _normalize(df)
            write_pickle(df, p)
            results[(label, method)] = df
            print('Saved:', p.name, 'rows', len(df))
        except Exception as e:
            errors.append(
                {
                    'scenario': label,
                    'input_db': input_db,
                    'output_db': output_db,
                    'method': method,
                    'error': repr(e),
                }
            )
            print('Skip', label, method, '->', repr(e))

# Summary table: outcome profile per method and scenario
summary_rows = []
for scenario in SCENARIOS:
    label = str(scenario['label'])
    input_db = str(scenario['input_db'])
    output_db = str(scenario['output_db'])
    ids = [str(x) for x in (scenario['ids'] or [])]

    for method in METHODS:
        df = results.get((label, method), pd.DataFrame())
        counts = _counts_1_to_0_1_to_1_1_to_n(df, ids)
        summary_rows.append(
            {
                'scenario': label,
                'input_db': input_db,
                'output_db': output_db,
                'method': method,
                'n_inputs': len(set(ids)),
                **counts,
            }
        )

summary = pd.DataFrame(summary_rows)
if not summary.empty:
    summary['frac_1_to_0'] = summary['1→0'] / summary['n_inputs'].replace(0, pd.NA)
    summary['frac_1_to_1'] = summary['1→1'] / summary['n_inputs'].replace(0, pd.NA)
    summary['frac_1_to_n'] = summary['1→n'] / summary['n_inputs'].replace(0, pd.NA)
    summary = summary.sort_values(['scenario', 'output_db', 'method']).reset_index(drop=True)

summary_path = CACHE_DIR / 'comparison_external_mapper_outcome_summary.csv'
summary.to_csv(summary_path, index=False)
print('Wrote:', summary_path)

if errors:
    err_df = pd.DataFrame(errors)
    err_path = CACHE_DIR / 'comparison_external_mapper_errors.csv'
    err_df.to_csv(err_path, index=False)
    print('Wrote:', err_path)

summary


In [ ]:
# -------------------- Notes --------------------

# This notebook intentionally avoids running IDTrack conversions because graph loading can be memory-intensive.
# For IDTrack outcome-profile panels and drift diagnostics, see:
# - `experiment_random_stress_tests/00_random_stress_tests_fig4ab.ipynb`
# - `experiment_hlca/00_hlca_manuscript_table1_and_figures.ipynb`


In [ ]:
# -------------------- Build capability matrix (Figure 4d) --------------------

capabilities = [
    'Pin historical Ensembl release',
    'Snapshot boundary (history window)',
    'Assembly-aware build axis',
    'Explainable / auditable paths',
    'External allowlist as contract',
    'Offline rerunnable after caching',
]

tools = ['IDTrack', 'pybiomart', 'mygene', 'g:Profiler', 'gget']

# Conservative, capability-first matrix.
cap = pd.DataFrame(False, index=tools, columns=capabilities)
cap.loc['IDTrack', :] = [True, True, True, True, True, True]
cap.loc['pybiomart', 'Pin historical Ensembl release'] = True

mat = cap.astype(int)

fig = plt.figure(figsize=(14, 4.8), constrained_layout=True)

gs = fig.add_gridspec(1, 3, width_ratios=[1.65, 1.0, 1.0])
ax0 = fig.add_subplot(gs[0, 0])
ax1 = fig.add_subplot(gs[0, 1])
ax2 = fig.add_subplot(gs[0, 2])

if sns is not None:
    sns.heatmap(
        mat,
        ax=ax0,
        cmap=['#FFFFFF', MANUSCRIPT_COLORS['1→1']],
        cbar=False,
        linewidths=0.5,
        linecolor=MANUSCRIPT_COLORS['grid'],
    )
else:
    ax0.imshow(mat.values)

ax0.set_title('Capability matrix (not an accuracy benchmark)')
ax0.set_xlabel('')
ax0.set_ylabel('')
ax0.set_xticklabels(ax0.get_xticklabels(), rotation=25, ha='right')

def _plot_demo(ax, scenario_label: str, title: str) -> None:
    if 'summary' not in globals() or summary is None or summary.empty:
        ax.axis('off')
        ax.text(0.5, 0.5, 'External-mapper demo not available', ha='center', va='center')
        return

    sub = summary[summary['scenario'] == scenario_label]
    if sub.empty:
        ax.axis('off')
        ax.text(0.5, 0.5, f'No rows for {scenario_label}', ha='center', va='center')
        return

    s = sub.set_index('method')[['1→0', '1→1', '1→n']]
    s_norm = s.div(s.sum(axis=1), axis=0)
    s_norm.plot(
        kind='bar',
        stacked=True,
        ax=ax,
        color=[MANUSCRIPT_COLORS['1→0'], MANUSCRIPT_COLORS['1→1'], MANUSCRIPT_COLORS['1→n']],
    )
    ax.set_ylim(0, 1)
    ax.set_ylabel('Fraction of queries')
    ax.set_title(title)
    ax.legend(['1→0', '1→1', '1→n'], loc='upper right', frameon=True)


_plot_demo(ax1, 'Ensembl→HGNC', 'Outcome demo: Ensembl→HGNC')
_plot_demo(ax2, 'HGNC→Ensembl', 'Outcome demo: HGNC→Ensembl')

written = save_figure(fig, 'fig_tool_comparison_matrix.pdf', ctx, formats=('pdf',))
print('Saved:', written['pdf'])


# Marketing extension: export the capability matrix as a table

The heatmap is the manuscript panel, but a table export is useful for:

- quick iteration while writing Results text
- reviewer responses (explicit checklists)
- supplementary-style reporting (if needed later)


In [ ]:
from experiments_utils import atomic_write_dataframe_csv, atomic_write_dataframe_latex  # noqa: E402

cap_table = mat.copy()
cap_table.insert(0, 'Tool', cap_table.index)
cap_table = cap_table.reset_index(drop=True)

out_csv = ctx.manuscript_tables / 'tool_comparison_capability_matrix.csv'
atomic_write_dataframe_csv(cap_table, out_csv, index=False)
atomic_write_dataframe_csv(cap_table, ctx.experiment_outputs / 'tables' / out_csv.name, index=False)
print('Wrote:', out_csv)

out_tex = ctx.manuscript_tables / 'tool_comparison_capability_matrix.tex'
atomic_write_dataframe_latex(cap_table, out_tex, index=False)
atomic_write_dataframe_latex(cap_table, ctx.experiment_outputs / 'tables' / out_tex.name, index=False)
print('Wrote:', out_tex)
cap_table
